# RetailIQ - Phase 4: Feature Engineering

## Overview
This notebook covers **Mission 4** of the RetailIQ analytics platform: developing feature engineering pipelines and outputs to support three specific machine learning tasks in Phase 5:
1. **Sales Forecasting (Time-Series):** Daily aggregate sales and revenue prediction.
2. **Customer Segmentation (Clustering):** Client RFM profile and behavioral metrics.
3. **Product Demand Prediction (Regression/Classification):** Weekly unit sales forecasting per product code.

### Crucial Constraint & Handlings:
- **Negative Tenure & Pre-Registration Sales:** In the customer RFM table, we flag the 5 customers with negative tenure (`IsNegativeTenure = 1`) and replace their `Tenure` value with the median tenure of valid customers (`902` days) to prevent invalid values from corrupting clustering algorithms.

## 1. Environment Setup & Data Loading
We load the libraries and cleaned database tables.

In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = os.path.join("..", "data", "processed", "retailiq.db")
PROCESSED_DIR = os.path.join("..", "data", "processed")

conn = sqlite3.connect(DB_PATH)
sales = pd.read_sql_query("SELECT * FROM clean_sales;", conn)
customers = pd.read_sql_query("SELECT * FROM clean_customers;", conn)
products = pd.read_sql_query("SELECT * FROM clean_products;", conn)
stores = pd.read_sql_query("SELECT * FROM clean_stores;", conn)
conn.close()

sales['Date'] = pd.to_datetime(sales['Date'])
customers['JoinDate'] = pd.to_datetime(customers['JoinDate'])
print("Setup complete. Datasets loaded.")

Setup complete. Datasets loaded.


## 2. Shared / Base Features
We extract shared time and order value markers from the transactional data:
- **Weekend Order:** Boolean flag indicating if order weekday is Saturday or Sunday.
- **Holiday/Festival Season Flag:** Active during major shopping windows (New Year, Independence Day, Thanksgiving, Christmas/Holiday season).
- **Order Value Tier:** Categorical tier (Low/Medium/High) based on quantiles (33% and 66% boundaries: $1352.25 and $3756.94).
- **High Value Order flag:** Boolean indicator for the top 10% highest orders (threshold: $6694.76).

In [2]:
sales['IsWeekend'] = sales['Date'].dt.weekday.isin([5, 6]).astype(int)

def get_holiday_season(dates):
    is_holiday = []
    for d in dates:
        y = d.year
        ny_start, ny_end = pd.Timestamp(f'{y-1}-12-31'), pd.Timestamp(f'{y}-01-02')
        id_start, id_end = pd.Timestamp(f'{y}-07-03'), pd.Timestamp(f'{y}-07-05')
        tg_start, tg_end = pd.Timestamp(f'{y}-11-20'), pd.Timestamp(f'{y}-11-30')
        xm_start, xm_end = pd.Timestamp(f'{y}-12-15'), pd.Timestamp(f'{y}-12-26')
        
        if (
            (d >= ny_start and d <= ny_end) or
            (d >= id_start and d <= id_end) or
            (d >= tg_start and d <= tg_end) or
            (d >= xm_start and d <= xm_end)
        ):
            is_holiday.append(1)
        else:
            is_holiday.append(0)
    return is_holiday

sales['IsHolidaySeason'] = get_holiday_season(sales['Date'])

q33, q66, q90 = 1352.25, 3756.94, 6694.76
sales['OrderValueTier'] = pd.cut(sales['OrderValue'], bins=[0, q33, q66, np.inf], labels=['Low', 'Medium', 'High'])
sales['IsHighValueOrder'] = (sales['OrderValue'] > q90).astype(int)

print('Base features applied on clean_sales.')
print('OrderValue quantiles thresholds used:')
print(f'33% (Low/Medium): ${q33}')
print(f'66% (Medium/High): ${q66}')
print(f'90% (High Value Order Flag): ${q90}')

Base features applied on clean_sales.
OrderValue quantiles thresholds used:
33% (Low/Medium): $1352.25
66% (Medium/High): $3756.94
90% (High Value Order Flag): $6694.76


## 3. Sales Forecasting Dataset (Time-Series)
We aggregate transactional sales data into a daily time series of total revenue and quantities sold. We reindex to a complete 731-day date range to fill missing calendar days with 0 (ensuring mathematically correct lags).
- **Features Derived:** Day of week, Month, Quarter, Year, Weekend indicator, Holiday season indicator.
- **Time Series Features:** Lag 7 days, Lag 30 days, 7-day rolling mean, 30-day rolling mean.

In [3]:
full_dates = pd.date_range(start=sales['Date'].min(), end=sales['Date'].max(), freq='D')
daily_sales = sales.groupby('Date').agg(
    TotalRevenue=('OrderValue', lambda x: (x * (1 - sales.loc[x.index, 'Discount'])).sum()),
    TotalVolume=('Quantity', 'sum')
).reindex(full_dates, fill_value=0.0).reset_index()
daily_sales.rename(columns={'index': 'Date'}, inplace=True)

daily_sales['DayOfWeek'] = daily_sales['Date'].dt.weekday
daily_sales['Month'] = daily_sales['Date'].dt.month
daily_sales['Quarter'] = daily_sales['Date'].dt.quarter
daily_sales['Year'] = daily_sales['Date'].dt.year
daily_sales['IsWeekend'] = daily_sales['Date'].dt.weekday.isin([5, 6]).astype(int)
daily_sales['IsHolidaySeason'] = get_holiday_season(daily_sales['Date'])

for lag in [7, 30]:
    daily_sales[f'Revenue_Lag_{lag}'] = daily_sales['TotalRevenue'].shift(lag)
    daily_sales[f'Volume_Lag_{lag}'] = daily_sales['TotalVolume'].shift(lag)

for window in [7, 30]:
    daily_sales[f'Revenue_RollMean_{window}'] = daily_sales['TotalRevenue'].rolling(window=window).mean()
    daily_sales[f'Volume_RollMean_{window}'] = daily_sales['TotalVolume'].rolling(window=window).mean()

daily_sales.to_csv(os.path.join(PROCESSED_DIR, "forecast_dataset.csv"), index=False)
print("=== Sales Forecasting Dataset ===")
print(f"Shape: {daily_sales.shape}")
print("\nColumns:")
print(", ".join(daily_sales.columns))
print("\nFirst 5 Rows:")
display(daily_sales.head())

=== Sales Forecasting Dataset ===
Shape: (731, 17)

Columns:
Date, TotalRevenue, TotalVolume, DayOfWeek, Month, Quarter, Year, IsWeekend, IsHolidaySeason, Revenue_Lag_7, Volume_Lag_7, Revenue_Lag_30, Volume_Lag_30, Revenue_RollMean_7, Volume_RollMean_7, Revenue_RollMean_30, Volume_RollMean_30

First 5 Rows:


,Date,TotalRevenue,TotalVolume,DayOfWeek,Month,Quarter,Year,IsWeekend,IsHolidaySeason,Revenue_Lag_7,Volume_Lag_7,Revenue_Lag_30,Volume_Lag_30,Revenue_RollMean_7,Volume_RollMean_7,Revenue_RollMean_30,Volume_RollMean_30
0,2023-09-10,11735.474,13.0,6,9,3,2023,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-09-11,12562.104,9.0,0,9,3,2023,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-09-12,4811.477,8.0,1,9,3,2023,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-09-13,11285.173,11.0,2,9,3,2023,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-09-14,27733.183,32.0,3,9,3,2023,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Customer Segmentation Dataset (RFM + Demographics)
We build a client-level behavioral table using RFM markers. Recency is measured relative to a fixed baseline date `2025-09-10` (one day after the final transactional date in raw sales).
- **RFM Metrics:** Recency (days since last purchase), Frequency (order transaction count), Monetary (sum of post-discount spend).
- **Additional Metrics:** Average Order Value (AOV), Favorite Category (mode of categories purchased).
- **Anomalies Handling:** Exclude negative-tenure cases from tenure features by mapping `IsNegativeTenure = 1` and replacing Tenure with the sample median (902 days) for these 5 clients.

In [4]:
sales['Revenue'] = sales['OrderValue'] * (1 - sales['Discount'])
cust_rfm = sales.groupby('CustomerID').agg(
    Frequency=('TransactionID', 'count'),
    Monetary=('Revenue', 'sum'),
    AOV=('Revenue', 'mean'),
    MaxOrderDate=('Date', 'max')
).reset_index()

baseline = pd.Timestamp('2025-09-10')
cust_rfm['Recency'] = (baseline - cust_rfm['MaxOrderDate']).dt.days

sales_prod = sales.merge(products[['ProductID', 'Category']], on='ProductID', how='left')
fav_cat = sales_prod.groupby('CustomerID')['Category'].agg(lambda x: x.mode()[0]).reset_index()
fav_cat.rename(columns={'Category': 'FavoriteCategory'}, inplace=True)

cust_seg = customers.merge(cust_rfm, on='CustomerID', how='left')
cust_seg = cust_seg.merge(fav_cat, on='CustomerID', how='left')

median_tenure = cust_seg[cust_seg['Tenure'] >= 0]['Tenure'].median()
cust_seg['IsNegativeTenure'] = (cust_seg['Tenure'] < 0).astype(int)
cust_seg['Tenure'] = np.where(cust_seg['Tenure'] < 0, median_tenure, cust_seg['Tenure'])

cust_segmentation = cust_seg[[
    'CustomerID', 'Gender', 'City', 'Age', 'Tenure', 'IsNegativeTenure',
    'Recency', 'Frequency', 'Monetary', 'AOV', 'FavoriteCategory'
]]

cust_segmentation.to_csv(os.path.join(PROCESSED_DIR, "segmentation_dataset.csv"), index=False)
print("=== Customer Segmentation Dataset ===")
print(f"Shape: {cust_segmentation.shape}")
print("\nColumns:")
print(", ".join(cust_segmentation.columns))
print("\nFirst 5 Rows:")
display(cust_segmentation.head())

=== Customer Segmentation Dataset ===
Shape: (200, 11)

Columns:
CustomerID, Gender, City, Age, Tenure, IsNegativeTenure, Recency, Frequency, Monetary, AOV, FavoriteCategory

First 5 Rows:


,CustomerID,Gender,City,Age,Tenure,IsNegativeTenure,Recency,Frequency,Monetary,AOV,FavoriteCategory
0,C001,M,Osborneport,29,1077.0,0,4,32,94818.5490,2963.079656,Fashion
1,C002,M,New Gabrielleport,66,1699.0,0,73,25,47302.4240,1892.096960,Electronics
2,C003,F,Port Allen,21,545.0,0,31,26,88093.9470,3388.228731,Electronics
3,C004,M,East Edgarborough,34,362.0,0,4,24,53715.9865,2238.166104,Fashion
4,C005,M,Port Kimberly,34,1244.0,0,50,23,54701.0415,2378.306152,Electronics


## 5. Product Demand Prediction Dataset
We aggregate product sales to a **weekly level** (starting Monday) to resolve sparsity/zero-demand bias associated with daily product transactions. This matches standard supply chain inventory ordering cycles.
- **Features Derived:** Weekly units sold per product code (`UnitsSold`).
- **Time Series Features:** Demand Lag 1 week, Demand Lag 4 weeks, 4-week rolling average, 12-week rolling average.
- **Calendar Context:** Month, Quarter, Holiday season indicator of the week.

In [5]:
sales['Week_MON'] = sales['Date'] - pd.to_timedelta(sales['Date'].dt.weekday, unit='D')
all_products = products['ProductID'].unique()
all_weeks = pd.date_range(start=sales['Week_MON'].min(), end=sales['Week_MON'].max(), freq='W-MON')

index = pd.MultiIndex.from_product([all_products, all_weeks], names=['ProductID', 'Week'])
skeleton = pd.DataFrame(index=index).reset_index()

prod_weekly = sales.groupby(['ProductID', 'Week_MON'])['Quantity'].sum().reset_index()
prod_weekly.rename(columns={'Week_MON': 'Week', 'Quantity': 'UnitsSold'}, inplace=True)

demand_df = skeleton.merge(prod_weekly, on=['ProductID', 'Week'], how='left').fillna(0.0)
demand_df = demand_df.merge(products[['ProductID', 'Category', 'SubCategory']], on='ProductID', how='left')

demand_df.sort_values(by=['ProductID', 'Week'], inplace=True)

demand_df['Demand_Lag_1'] = demand_df.groupby('ProductID')['UnitsSold'].shift(1)
demand_df['Demand_Lag_4'] = demand_df.groupby('ProductID')['UnitsSold'].shift(4)
demand_df['Demand_RollMean_4'] = demand_df.groupby('ProductID')['UnitsSold'].transform(lambda x: x.rolling(4).mean())
demand_df['Demand_RollMean_12'] = demand_df.groupby('ProductID')['UnitsSold'].transform(lambda x: x.rolling(12).mean())

demand_df['Month'] = demand_df['Week'].dt.month
demand_df['Quarter'] = demand_df['Week'].dt.quarter
demand_df['IsHolidaySeason'] = get_holiday_season(demand_df['Week'])

demand_df.to_csv(os.path.join(PROCESSED_DIR, "demand_dataset.csv"), index=False)
print("=== Product Demand Prediction Dataset ===")
print(f"Shape: {demand_df.shape}")
print("\nColumns:")
print(", ".join(demand_df.columns))
print("\nFirst 5 Rows:")
display(demand_df.head())

=== Product Demand Prediction Dataset ===
Shape: (5300, 12)

Columns:
ProductID, Week, UnitsSold, Category, SubCategory, Demand_Lag_1, Demand_Lag_4, Demand_RollMean_4, Demand_RollMean_12, Month, Quarter, IsHolidaySeason

First 5 Rows:


,ProductID,Week,UnitsSold,Category,SubCategory,Demand_Lag_1,Demand_Lag_4,Demand_RollMean_4,Demand_RollMean_12,Month,Quarter,IsHolidaySeason
0,P001,2023-09-04,0.0,Electronics,Camera,NaN,NaN,NaN,NaN,9,3,0
1,P001,2023-09-11,3.0,Electronics,Camera,0.0,NaN,NaN,NaN,9,3,0
2,P001,2023-09-18,0.0,Electronics,Camera,3.0,NaN,NaN,NaN,9,3,0
3,P001,2023-09-25,3.0,Electronics,Camera,0.0,NaN,1.5,NaN,9,3,0
4,P001,2023-10-02,2.0,Electronics,Camera,3.0,0.0,2.0,NaN,10,4,0


## 6. Encoding & Scaling Plan (For Phase 5)

Here we document the preprocessing strategy for each of the engineered datasets to guide model training in Phase 5:

### 1. Sales Forecasting Dataset
- **Encoding:** 
  - Target encoding or Cyclic sine/cosine transformations on calendar variables (`Month`, `DayOfWeek`, `Quarter`) to capture periodic characteristics.
- **Scaling:** 
  - Apply **StandardScaler** or **RobustScaler** (if anomalies are present) on target variables (`TotalRevenue`, `TotalVolume`) and lag features (`Revenue_Lag_7`, `Volume_Lag_30`, etc.) when training linear models or neural networks (tree models do not require scaling).

### 2. Customer Segmentation Dataset
- **Encoding:** 
  - One-hot encode binary attributes (`Gender`).
  - One-hot or Target encoding on high-cardinality string categories (`City`, `FavoriteCategory`).
- **Scaling:** 
  - Apply **MinMaxScaler** or **StandardScaler** to numerical columns (`Age`, `Tenure`, `Recency`, `Frequency`, `Monetary`, `AOV`) prior to running distance-based clustering algorithms like **K-Means** or **DBSCAN**. Distance algorithms are highly sensitive to column magnitudes.

### 3. Product Demand Prediction Dataset
- **Encoding:**
  - Target encoding or One-Hot encoding on product identifiers (`ProductID`) and categorical grouping attributes (`Category`, `SubCategory`).
  - Cyclic sine/cosine encoding on week-calendar index (`Month`, `Quarter`).
- **Scaling:**
  - Apply **StandardScaler** to `UnitsSold` and corresponding lag/rolling statistics (`Demand_Lag_1`, `Demand_RollMean_12`) for regularization and optimization convergence in linear estimators (e.g. Ridge, Lasso) or neural nets.

## 7. Engineered Datasets Summary

| Dataset Name | File Path | Shape | Purpose / ML Model Target |
| :--- | :--- | :--- | :--- |
| **Sales Forecasting Dataset** | `data/processed/forecast_dataset.csv` | `(731, 17)` | Daily univariate/multivariate time-series forecasting of total revenue and transaction volumes. |
| **Customer Segmentation Dataset** | `data/processed/segmentation_dataset.csv` | `(200, 11)` | Client-level behavioral clustering (RFM) and store segment profiling. |
| **Product Demand Dataset** | `data/processed/demand_dataset.csv` | `(5300, 12)` | Weekly product code-level unit sales forecasting for supply chain stock optimization. |